<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### My Lane: Ranking Signal Analysis

**Question:** Which signals are associated with page performance?

**Task Type:** Signal analysis (primary) + Ranking (secondary)

### Why Three Models?

I use three models to get a complete picture:

| Model | What It Tells Me | Why It Fits My Lane |
|---|---|---|
| **Logistic Regression** | Coefficients (direction + magnitude of each signal) | Answers "which signals push the needle?" |
| **Decision Tree (depth 3)** | Interactions between signals (readable flowchart) | Answers "how do signals work together?" |
| **Random Forest** | Feature importance (which signals matter most) | Answers "what should we focus on?" |

### Why These Models Fit My Question

My lane asks: "Which signals are associated with page performance?"

- **Logistic Regression** → Shows if a signal increases or decreases decline risk
- **Decision Tree** → Shows how signals combine (e.g., age + position together)
- **Random Forest** → Shows which signals are most important overall

### What I'll Compare

All models will be compared against my Week 4 baseline on the same metric: **Precision@20** and **Precision@50**.

The baseline rule is: `score = stale × visible × impressions` (captures 17 pages).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Validation Strategy: Client-Holdout (80/20 split by client_id)

**Why client-holdout?**
- Prevents the model from memorizing client-specific patterns
- Tests whether the model can generalize to unseen clients
- Matches real-world deployment (new clients will be unseen)

**Why 80/20?**
- 30,000 rows → 24,000 train, 6,000 test
- Enough data to train well
- Enough test data to evaluate reliably
- Matches the starter pipeline's approach

---

### Data Preparation

**Features (all knowable at decision time):**

| Feature | Why It's a Feature |
|---|---|
| `avg_position` | Average search position — knowable from feature window |
| `ctr` | Click-through rate — knowable from feature window |
| `engagement_rate` | Visitor engagement — knowable from feature window |
| `content_age_days` | Age of content — knowable at decision point |
| `word_count` | Content length — knowable at decision point |
| `search_volume` | Keyword demand — knowable from feature window |
| `competition` | Competition level — knowable from feature window |
| `days_since_last_update` | Days since update — knowable at decision point |
| `content_type` | Type of page (categorical) — knowable at decision point |
| `main_intent` | User intent (categorical) — knowable at decision point |

**Label:**
- `is_declining_label` — whether the page is declining (observed outcome)

**Excluded (Leakage Prevention):**

| Column | Why Excluded |
|---|---|
| `trend_pct` | Derived from the label — LEAKAGE! |
| `trend_direction` | Derived from the label — LEAKAGE! |
| `content_id` | Identifier, NOT a signal |
| `client_id` | Used ONLY for splitting, NOT as a feature |

**Missing Value Handling:**
- `avg_position`: 0 means "no data" → replaced with -1
- Other numeric columns: filled with median
- Categorical columns: one-hot encoded

---

### Implementation

- 80% of clients → Training set
- 20% of clients → Test set
- Client_id used ONLY for splitting, NOT as a feature

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Setup for Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Loaded {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}\n")

print(df["impressions_90d"])
print(df["days_since_last_update"])
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
print(stale)
print(visible)

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded 30,000 rows
Declining rate: 0.542

0          3803
1         15320
2         12581
3         11751
4         19140
          ...  
29995         1
29996       761
29997      6336
29998    154763
29999      4234
Name: impressions_90d, Length: 30000, dtype: int64
0         20
1         25
2         20
3         22
4         14
        ... 
29995     20
29996     20
29997     20
29998     22
29999    104
Name: days_since_last_update, Length: 30000, dtype: int64
0        0
1        0
2        0
3        0
4        0
        ..
29995    0
29996    0
29997    0
29998    0
29999    0
Name: days_since_last_update, Length: 30000, dtype: int64
0        1
1        1
2        1
3        1
4        1
        ..
29995    0
29996    1
29997    1
29998    1
29999    1
Name: impressions_90d, Length: 30000, dtype: int64


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.